::: {.sb-lesson-links}
[Exercises →](../exercises/california-housing.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/02-linear-regression.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-maximum-likelihood.ipynb){download="lesson-maximum-likelihood.ipynb"}  
Data: [california_housing_market.csv](../downloads/data/california_housing_market.csv){download="california_housing_market.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

The derivations of this optional lesson are illustrated with small demos 🔬 on simulated and real data. The next cell loads the libraries.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats, optimize
from sklearn.linear_model import LinearRegression
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"
np.set_printoptions(suppress=True)   # print arrays without scientific notation

## Definition

The likelihood of a family of random variables is a function that gives, for each possible combination of realizations of these random variables, the probability that this combination occurs, given the parameters of the model. In statistical modeling, the data are already available, so we already know the realization of each random variable (i.e. the values of the variables for each observation). The goal is therefore to find the parameters that make the observations we have as likely as possible. In other words, we are trying to find the general rule that we suspect exists behind the observed distribution of the data. 

Statistical likelihood is a function of conditional probabilities (a probability whose distribution depends on parameters). Let $X = (X_{1}, X_{2}, ..., X_{n})$ be a vector of random variables and $\theta = (\beta_{1}, \beta_{2}, ..., \beta_{k})$ the set of parameters on which $X$ depends. In what follows, capital letters represent the random variables, i.e. $X_i$, and lowercase letters represent specific realizations of those random variables, $X_i = x_i$. With that in mind, the likelihood of $Y$ is written:

$$
L(Y)=L(x_{1},x_{2},...x_{p}) = \prod^{n}_{i=1}f(x_{i};\beta)
$$

With

if $x$ is a continuous variable:

$$
f(x_{i};\beta) = f_{\beta}(x_{i})
$$

where $f$ is the probability density function of $X_i$, and $\beta$ represents the parameters $\beta_1, \beta_2, ..., \beta_p$.

if $x$ is a discrete variable:

$$
f(x_{i};\theta) = P_{\beta}(x_{i})
$$

where $P$ is the probability of getting $X_i = x_i$.



::: {.callout-tip title="Intuition"}
Likelihood turns the usual question around. Probability asks: "with these parameters, how likely are these data?". Likelihood asks: "given the data I observed, which parameters make them the most plausible?". Like a detective, we try candidate explanations (parameter values) and keep the one under which what we observed is the least surprising.
:::



#### See it with data 🔬: finding the most likely mean

We measure the height of 20 people (simulated data). We assume heights follow a normal distribution with a standard deviation of 8 cm, but we don't know its mean $\mu$. For each candidate value of $\mu$, we compute the likelihood: the product of the 20 densities.

In [ ]:
rng = np.random.default_rng(7)
heights = rng.normal(170, 8, 20).round(1)

def likelihood(mu):
    return np.prod(stats.norm.pdf(heights, loc=mu, scale=8))

for mu in [160, 165, 170, 175]:
    print(f"mu = {mu} cm -> likelihood = {likelihood(mu):.3e}")

mu = 160 cm -> likelihood = 4.449e-33
mu = 165 cm -> likelihood = 1.057e-29
mu = 170 cm -> likelihood = 1.017e-29
mu = 175 cm -> likelihood = 3.957e-33


In [ ]:
mus = np.linspace(160, 180, 201)
log_lik = [np.sum(stats.norm.logpdf(heights, loc=mu, scale=8)) for mu in mus]
best_mu = mus[np.argmax(log_lik)]

fig = px.line(x=mus, y=log_lik, title="Log-likelihood of the heights for different values of the mean",
              labels={"x": "Candidate mean mu (cm)", "y": "Log-likelihood"})
fig.add_vline(x=heights.mean(), line_dash="dash", annotation_text="sample mean")
fig.show()
print(f"mu with the highest likelihood: {best_mu:.1f} cm   sample mean: {heights.mean():.2f} cm")

mu with the highest likelihood: 167.5 cm   sample mean: 167.47 cm


Two things to notice:

* The likelihoods are tiny numbers (around $10^{-29}$ at best), because we multiply 20 densities that are all smaller than 1. With thousands of observations, the product would become too small for the computer to store. This is a very practical reason to take the logarithm, which turns the product into a sum, as in the derivation below.
* The log-likelihood has a single peak, at $\mu$ = 167.5 cm, which is the sample mean (167.47 cm). For a normal distribution, the maximum likelihood estimate of the mean is simply the average of the observations.

### Maximum likelihood estimate for simple linear regression

One way to estimate the parameters of a model is to maximize the corresponding likelihood function. In the case of simple linear regression, this function is:

$$
L(Y = y|\beta) = \prod_{i=1}^{n}f(Y_{i} = y_{i}|\beta)
$$

$$
L(Y = y|\beta) = \prod_{i=1}^{n}f(\beta_{0} + \beta_1 x_{i} + \epsilon = y_{i}|\beta_{0}, \beta_{1})
$$

$$
L(Y = y|\beta) = \prod_{i=1}^{n}f(\epsilon = y_{i}-\beta_{0}-\beta_{1}x_{i}|\beta_{0}, \beta_{1})
$$

$$
L(Y = y|\beta) = \prod_{i=1}^{n}\frac{1}{\sigma\sqrt{2\pi}}\exp(\frac{-(y_{i}-\beta_{0}-\beta_{1}x_{i})^2}{2\sigma^2})
$$

This likelihood equation is obtained by assuming that the error $\epsilon$ follows a centered normal distribution ($E(\epsilon) = \mu = 0$) with standard deviation $\sigma$. We must therefore find the maximum (if it exists) of this likelihood function. To do this, we apply a logarithm to each side of the equation, to get a sum rather than a product.

$$
log(L(Y = y|\beta)) = log(\prod^{n}_{i=1}\frac{1}{\sigma \sqrt{2\pi}}\exp(\frac{-(y_{i}-\beta_{0}-\beta_{1}x_{i})^2}{2\sigma^2}))
$$

$$
l(Y = y|\beta) = -n(log(\sigma)+\frac{1}{2}log(2\pi))-\frac{1}{2\sigma^2}\sum^{n}_{i=1}(y_{i}-\beta_{0}-\beta_{1}x_{i})^2
$$



We can see that the only part of this equation that depends on the parameters $β_0$ and $β_1$ is the sum of squared errors. We still have to find the values for which $-\sum^{n}_{i=1}(y_{i}-\beta_{0}-\beta_{1}x_{i})^2$ is maximal, i.e. the values that minimize the sum of squared errors, exactly as in the least squares method.

#### See it with data 🔬: maximum likelihood and least squares find the same line

We simulate 100 points with $y = 1 + 2x + \epsilon$, where $\epsilon$ follows a normal distribution with $\sigma = 1.5$. Then we compute the log-likelihood (with the formula above) and the sum of squared errors for many values of $\beta_1$, keeping $\beta_0$ and $\sigma$ fixed.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 100)
y = 1 + 2 * x + rng.normal(0, 1.5, 100)
ols = LinearRegression().fit(x.reshape(-1, 1), y)
b0, sigma, n = ols.intercept_, 1.5, len(y)

def log_likelihood(b1):
    sse = np.sum((y - b0 - b1 * x) ** 2)
    return -n * (np.log(sigma) + 0.5 * np.log(2 * np.pi)) - sse / (2 * sigma**2)

b1_grid = np.linspace(1.8, 2.2, 401)
ll = [log_likelihood(b) for b in b1_grid]
print(f"beta_1 with the highest log-likelihood: {b1_grid[np.argmax(ll)]:.3f}")
print(f"beta_1 found by least squares        : {ols.coef_[0]:.3f}")

beta_1 with the highest log-likelihood: 1.980
beta_1 found by least squares        : 1.980


In [ ]:
fig = px.line(x=b1_grid, y=ll, title="Log-likelihood of the simulated data as a function of beta_1",
              labels={"x": "beta_1", "y": "Log-likelihood"})
fig.add_vline(x=ols.coef_[0], line_dash="dash", annotation_text="least squares estimate")
fig.show()

The log-likelihood is an upside-down bowl, and its peak ($\beta_1$ = 1.980) is exactly the slope found by least squares (1.980), close to the true value 2. This is the equivalence shown by the derivation: maximizing the log-likelihood means minimizing the sum of squared errors.

We fixed $\beta_0$ and $\sigma$ to draw the curve. Let's now let a numerical optimizer (`scipy.optimize.minimize`) search for the three parameters $\beta_0$, $\beta_1$ and $\sigma$ at the same time, by minimizing minus the log-likelihood.

In [ ]:
def neg_log_likelihood(params):
    b0, b1, log_sigma = params              # we optimize log(sigma) so that sigma stays positive
    return -np.sum(stats.norm.logpdf(y, loc=b0 + b1 * x, scale=np.exp(log_sigma)))

result = optimize.minimize(neg_log_likelihood, x0=[0, 0, 0])
b0_mle, b1_mle, sigma_mle = result.x[0], result.x[1], np.exp(result.x[2])
print(f"Maximum likelihood: beta_0 = {b0_mle:.4f}, beta_1 = {b1_mle:.4f}, sigma = {sigma_mle:.4f}")
print(f"Least squares     : beta_0 = {ols.intercept_:.4f}, beta_1 = {ols.coef_[0]:.4f}")
print(f"sqrt(SSR / n)     : {np.sqrt(np.mean((y - ols.predict(x.reshape(-1, 1))) ** 2)):.4f}")

Maximum likelihood: beta_0 = 0.9888, beta_1 = 1.9804, sigma = 1.4543
Least squares     : beta_0 = 0.9888, beta_1 = 1.9804
sqrt(SSR / n)     : 1.4543


The optimizer, which knows nothing about least squares, finds the same coefficients as `LinearRegression` ($\beta_0$ = 0.9888, $\beta_1$ = 1.9804). As a bonus, it also estimates the noise level: $\sigma$ = 1.4543 (the true value is 1.5), which is exactly $\sqrt{SSR/n}$, the typical size of the residuals.

### Maximum likelihood estimate for multiple linear regression

Computing the maximum likelihood estimator is a classic exercise in statistics. We present the calculation here for those who are familiar with matrix calculus, or would like to become familiar with it.


$$
L(Y = y|\beta) = f(Y=y|\beta)
$$

$$
L(Y = y|\beta) = f(X\beta+\epsilon = y|\beta)
$$

$$
L(Y = y|\beta) = f(\epsilon = y-X\beta|\beta)
$$

Under the assumptions of the multiple linear model, $\epsilon$ follows a centered normal distribution $(E(\epsilon) = 0_n)$ with a diagonal covariance matrix $\sum = Diag(\sigma^2, \sigma^{2}, ... , \sigma^{2}) = \sigma^2 I_n$ (homoscedasticity and independence of the residuals). This brings us to the following equation:

$$
L(Y = y|\beta) = det(2\pi\sum)^{-\frac{1}{2}}\exp(-\frac{1}{2}((y-X\beta)^{t}\sum^{-1}(y-X\beta)))
$$

$\sum = \sigma^2 I_n$ is proportional to the identity matrix, hence:

$$
L(Y = y|\beta) = det(2\pi\sum)^{-\frac{1}{2}}\exp(-\frac{1}{2}\sum^{-1}((y-X\beta)^{t}(y-X\beta)))
$$

We apply the logarithm, which is an increasing function and therefore does not change the optimization problem:

$$
log(L(Y = y|\beta)) = -\frac{1}{2}log(det(2\pi\sum))-\frac{1}{2}\sum^{-1}(y-X\beta)^{t}(y-X\beta)
$$


We look for the value of $\beta$ that maximizes the above equation, which is the same as minimizing the following quantity:

$$
\underset{\beta}{\arg\min}\ (y-X\beta)^{t}(y-X\beta) = \beta_{MLE}
$$

$$
\underset{\beta}{\arg\min}\ y^{t}y-\beta^{t}X^{t}y-y^{t}X\beta+\beta^{t}X^{t}X\beta = \beta_{MLE}
$$

$y^{t}X\beta$ is a scalar (i.e. a real number of dimension 1), so it is equal to its transpose: $y^{t}X\beta = \beta^{t}X^{t}y$. Hence:

$$
\underset{\beta}{\arg\min}\ y^{t}y-2\beta^{t}X^{t}y + \beta^{t}X^{t}X\beta = \beta_{MLE}
$$

We differentiate with respect to $\beta$, set the derivative to zero, and get:

$$
-X^{t}y + X^{t}X\beta = 0 \Rightarrow \beta_{MLE} = (X^{t}X)^{-1}X^{t}y
$$

This solution is well defined only if $X^{t}X$ is an invertible matrix, which is the case if the explanatory variables are not collinear and if $p < n$ (the number of explanatory variables is less than the number of observations).

#### See it with data 🔬: the closed-form formula in numpy

Let's apply $\beta_{MLE} = (X^{t}X)^{-1}X^{t}y$ to the California housing dataset (8 explanatory variables plus the column of 1s), and compare with scikit-learn. In numpy, rather than computing the inverse, we solve the system $X^{t}X\beta = X^{t}y$ with `np.linalg.solve`, which is faster and more accurate.

In [ ]:
housing = pd.read_csv("../data/california_housing_market.csv")
features, target = housing.drop(columns="MedHouseVal"), housing["MedHouseVal"]
X = np.column_stack([np.ones(len(features)), features])

beta_mle = np.linalg.solve(X.T @ X, X.T @ target)
sk = LinearRegression().fit(features, target)
print("Formula :", beta_mle.round(4))
print("sklearn :", np.r_[sk.intercept_, sk.coef_].round(4))

Formula : [-36.9419   0.4367   0.0094  -0.1073   0.6451  -0.      -0.0038  -0.4213
  -0.4345]
sklearn : [-36.9419   0.4367   0.0094  -0.1073   0.6451  -0.      -0.0038  -0.4213
  -0.4345]


And when the conditions of the last sentence are not met? With only 5 districts for 9 coefficients ($p \geq n$), $X^{t}X$ is a $9 \times 9$ matrix whose rank can't exceed 5, so it is not invertible:

In [ ]:
X_small = X[:5]
XtX = X_small.T @ X_small
print("Shape of X^t X:", XtX.shape, "  rank:", np.linalg.matrix_rank(XtX))
inverse = np.linalg.inv(XtX)      # numpy returns numbers, but are they an inverse?
print("Does inverse @ XtX give the identity matrix?", np.allclose(inverse @ XtX, np.eye(9)))
print(f"Condition number of XtX: {np.linalg.cond(XtX):.1e}")

Shape of X^t X: (9, 9)   rank: 5
Does inverse @ XtX give the identity matrix? False
Condition number of XtX: 3.6e+22


On the full dataset, the formula gives the same 9 coefficients as scikit-learn. With 5 rows only, $X^{t}X$ has rank 5 instead of 9: it is not invertible. Numpy still returns a matrix, but it is not an inverse (multiplying it by $X^{t}X$ does not give the identity matrix), and the huge condition number (3.6e+22 here) is the warning sign that the computation is meaningless. Exactly collinear variables lead to the same situation. Scikit-learn does not use this formula directly: it relies on more robust methods that still return a solution (one among infinitely many) when $X^{t}X$ is not invertible.

::: {.sb-next}
**[Next: Linear regression with scikit-learn →](../lessons/workflow.ipynb)**
:::